# Convert OpenBerg NetCDF files to Shapefiles

*Nick Hughes, 2026-i-21*

The [OpenBerg iceberg drift model](https://opendrift.github.io/autoapi/opendrift/models/openberg/index.html) used in the [MET Norway Værio portal](https://xn--vrio-voa.no/trajectory-models) creates a NetCDF file with a set of latitude-longitude points representing trajectories and ancillary inforation, which is not possible to visualize in GIS. The simulation also provides a PNG quicklook showing the final point cloud and in which it is difficult to see individual tracks.

<img src="attachment:12961824-f619-4540-b125-eff581c6822b.png" alt="OpenBerg 2026-01-21 Example" width="300"/>

This notebook reads the NetCDF file and generates point, linestring, and polygon Shapefiles for use in GIS. These contain:

- Trajectory point locations with the ancillary data such as forcings and cboundary conditions provided by the model.

- A set of lines linking each set of trajectory points.

- Polygons mapping the concave hull around the point locations at each time step.

In [1]:
import os

import numpy as np

from netCDF4 import Dataset, num2date

import osgeo.ogr as ogr
import osgeo.osr as osr

In [2]:
ogr.UseExceptions()

## Input file name

In [3]:
ncpath = "/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/"+"iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end.nc"

## Split path and create output filenames

In [4]:
outdir, ncfname = os.path.split( ncpath )
ncroot, ncextn = os.path.splitext ( ncfname)
dirbits = outdir.split('/')
subdir = dirbits[-1]

In [5]:
pts_shppath = f"{outdir}/{ncroot}_{subdir}_points.shp"
print (pts_shppath)
pts_layname = f"{ncroot}_{subdir}_points"

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_points.shp


In [6]:
lns_shppath = f"{outdir}/{ncroot}_{subdir}_lines.shp"
print (lns_shppath)
lns_layname = f"{ncroot}_{subdir}_lines"

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_lines.shp


In [7]:
poly_shppath = f"{outdir}/{ncroot}_{subdir}_polygons.shp"
print (poly_shppath)
poly_layname = f"{ncroot}_{subdir}_polygons"

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_polygons.shp


In [8]:
shplist = [ pts_shppath, lns_shppath, poly_shppath ]

## Get NetCDF dimensions and list of variables

In [9]:
nc = Dataset( ncpath, 'r' )

In [10]:
nc = Dataset(ncpath, 'r')
nc.set_auto_mask(False)
print(np.nanmin(nc['lon'][:]))
print(np.nanmax(nc['lon'][:]))

print(np.nanmin(nc['lat'][:]))
print(np.nanmax(nc['lat'][:]))

print(np.nanmin(nc['z'][:]))

-40.09697
-27.687902
64.38752
67.71491
0.0


### Get dimensions

In [11]:
print( [ name for name in nc.dimensions.keys() ] )

['trajectory', 'time']


In [12]:
ntraj = nc.dimensions['trajectory'].size
ntime = nc.dimensions['time'].size
print (ntraj,ntime)

20 480


### Get list of variables

In [13]:
var_names = [ name for name in nc.variables.keys()]

Get trajectory and time lists

In [14]:
trajectories = np.array( nc['trajectory'][:], dtype=np.uint16 )

In [15]:
nctime = nc['time']
times = num2date( nctime, nctime.units, nctime.calendar)

In [16]:
ages = np.array( nc['age_seconds'][0,:], dtype=np.float32 )

In [17]:
var_dtypes = {}

for var_name in var_names:

    dtype = nc.variables[var_name].dtype
    dims = nc.variables[var_name].dimensions

    if len(dims) == 2:
        var_dtypes[var_name] = dtype.name

## Create new Shapefiles and configure

In [18]:
shpdrv = ogr.GetDriverByName("ESRI Shapefile")

In [19]:
for shppath in shplist:
    if os.path.exists(shppath):
         shpdrv.DeleteDataSource(shppath)

Define spatial reference.

In [20]:
ll_srs = osr.SpatialReference()
ll_srs.ImportFromEPSG(4326)
ll_srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

### Points Shapefile

In [21]:
pts_ds = shpdrv.CreateDataSource(pts_shppath)
pts_lay = pts_ds.CreateLayer( pts_layname, ll_srs, geom_type=ogr.wkbPoint )

### Line Shapefile

In [22]:
lns_ds = shpdrv.CreateDataSource(lns_shppath)
lns_lay = lns_ds.CreateLayer( pts_layname, ll_srs, geom_type=ogr.wkbLineString )

### Polygon Shapefile

In [23]:
poly_ds = shpdrv.CreateDataSource(poly_shppath)
poly_lay = lns_ds.CreateLayer( poly_layname, ll_srs, geom_type=ogr.wkbPolygon )

Add data fields

In [24]:
trajdef = ogr.FieldDefn( 'trajectory', ogr.OFTInteger )
pts_lay.CreateField( trajdef )
lns_lay.CreateField( trajdef )

0

In [25]:
timedef = ogr.FieldDefn( 'datetime', ogr.OFTString )
timedef.SetWidth(19)
pts_lay.CreateField( timedef )
poly_lay.CreateField( timedef )

0

In [26]:
dtype2oft = { 'int32': ogr.OFTInteger,
              'float32': ogr.OFTReal }

Shapefile field names can only be a limited number of characters, so provide a dictionary to translate from NetCDF.

In [27]:
shortnames = {
    'status':                                   'status',
    'moving':                                   'moving',
    'age_seconds':                              'age',
    'origin_marker':                            'origin',
    'lon':                                      'lon',
    'lat':                                      'lat',
    'z':                                        'z',
    'sail':                                     'sail',
    'draft':                                    'draft',
    'length':                                   'length',
    'width':                                    'width',
    'weight_coef':                              'weight',
    'water_form_drag_coef':                     'waterfdrag',
    'water_skin_drag_coef':                     'watersdrag',
    'wind_form_drag_coef':                      'windfdrag',
    'wind_skin_drag_coef':                      'windsdrag',
    'iceb_x_velocity':                          'ibu',
    'iceb_y_velocity':                          'ibv',
    'x_sea_water_velocity':                     'swu',
    'y_sea_water_velocity':                     'swv',
    'sea_floor_depth_below_sea_level':          'depth',
    'sea_surface_height':                       'ssh',
    'sea_surface_x_slope':                      'ssxslope',
    'sea_surface_y_slope':                      'ssyslope',
    'x_wind':                                   'windu',
    'y_wind':                                   'windv',
    'sea_surface_wave_significant_height':      'waveh',
    'sea_surface_wave_from_direction':          'wavedir',
    'sea_surface_wave_stokes_drift_x_velocity': 'stokesu',
    'sea_surface_wave_stokes_drift_y_velocity': 'stokesv',
    'sea_water_temperature':                    'sst',
    'sea_water_salinity':                       'sss',
    'sea_ice_area_fraction':                    'sic',
    'sea_ice_thickness':                        'sih',
    'sea_ice_x_velocity':                       'siu',
    'sea_ice_y_velocity':                       'siv',
    'land_binary_mask':                         'landmask',
    'wave_drag_coef':                           'wavedrag',
    'horizontal_diffusivity':                   'hordiff',
    }

In [28]:
for fieldname in var_dtypes:
    # print (fieldname, shortnames[fieldname], var_dtypes[fieldname], dtype2oft[var_dtypes[fieldname]] )
    fielddef = ogr.FieldDefn( shortnames[fieldname], dtype2oft[var_dtypes[fieldname]] )
    pts_lay.CreateField( fielddef )

Add line fields

In [29]:
startdef = ogr.FieldDefn( 'starttime', ogr.OFTString )
startdef.SetWidth(19)
lns_lay.CreateField( startdef )

enddef = ogr.FieldDefn( 'endtime', ogr.OFTString )
enddef.SetWidth(19)
lns_lay.CreateField( enddef )

0

Add polygon fields

In [30]:
fiddef = ogr.FieldDefn( 'fid', ogr.OFTInteger )
poly_lay.CreateField( fiddef )
agedef = ogr.FieldDefn( 'age', ogr.OFTReal )
poly_lay.CreateField( agedef )
areadef = ogr.FieldDefn( 'area', ogr.OFTReal )
poly_lay.CreateField( areadef )
peridef = ogr.FieldDefn( 'perimeter', ogr.OFTReal )
poly_lay.CreateField( peridef )

0

## Loop through trajectories and add to Shapefiles

In [31]:
for trajectory in trajectories:

    idx = 0

    lons = nc['lon'][trajectory,:]
    lats = nc['lat'][trajectory,:]
    zs = nc['z'][trajectory,:]
    # print (lons.shape,lats.shape,zs.shape)

    for time,lon,lat,z in zip(times,lons,lats,zs):
        # print (trajectory,time,lon,lat,z)
        dtstr = f"{time.strftime('%Y-%m-%d %H:%M:%S')}"

        if idx == 0:
            startdt = dtstr

            # Create line feature
            lns_feat = ogr.Feature(lns_lay.GetLayerDefn())

            # Create a line geometry
            line = ogr.Geometry(ogr.wkbLineString)

        # Create point feature
        pts_feat = ogr.Feature(pts_lay.GetLayerDefn())

        # Add fields to point feature
        pts_feat.SetField( 'trajectory', int(trajectory) )
        pts_feat.SetField( 'datetime', dtstr )
        for fieldname in var_dtypes:
            data = nc[fieldname][:]
            # if np.ma.isMaskedArray(data):
            #     print(fieldname)#, data.mask)
            #     print(
            #     trajectory,
            #     idx,
            #     nc['status'][trajectory])
            if var_dtypes[fieldname] == 'int32':
                pts_feat.SetField( shortnames[fieldname], int(nc[fieldname][trajectory,idx]))
            elif var_dtypes[fieldname] == 'float32':
                pts_feat.SetField( shortnames[fieldname], float(nc[fieldname][trajectory,idx]))
        # for fieldname in var_dtypes:
        #     value = nc[fieldname][trajectory, idx]
        #     if np.ma.is_masked(value):
        #         continue
        #     if var_dtypes[fieldname] == 'int32':
        #         pts_feat.SetField(
        #             shortnames[fieldname],
        #             int(value)
        #         )
        #     elif var_dtypes[fieldname] == 'float32':
        #         pts_feat.SetField(
        #             shortnames[fieldname],
        #             float(value)
        #         )
        # if not (np.isfinite(lon) and np.isfinite(lat) and np.isfinite(z)):
        #     print(
        #         f"Invalid coordinate: "
        #         f"trajectory={trajectory}, "
        #         f"idx={idx}, "
        #         f"lon={lon}, lat={lat}")

        # Create a point geometry
        if np.logical_or(np.isnan(lon), np.isnan(lat))==False:
            # print(lon,lat)
            point = ogr.Geometry(ogr.wkbPoint)
            point.AddPoint( float(lon), float(lat))#, float(z) )
            pts_feat.SetGeometry(point)


            # Add point to line
            line.AddPoint( float(lon), float(lat))#, float(z) )

            # Add feature to file
            pts_lay.CreateFeature(pts_feat)

        # Dereference the feature
        pts_feat = None

        idx = idx + 1
        # break

    # Add fields and geometry to line feature
    lns_feat.SetField( 'trajectory', int(trajectory) )
    lns_feat.SetField( 'starttime', startdt )
    lns_feat.SetField( 'endtime', dtstr )
    lns_feat.SetGeometry(line)

    # Add feature to file
    lns_lay.CreateFeature(lns_feat)

    # Dereference the feature
    lns_feat = None

    if trajectory % 100 == 0:
        print (f"{trajectory:5d} {startdt:19s} {dtstr:19s}")

    # if trajectory > 10:
    #     break

    0 2026-09-11 08:21:00 2026-10-01 07:21:00


In [32]:
print('done')

done


## Loop through ages and add to polygon Shapefile

Use concave hull to draw polygon around points.

In [33]:
hem_srs = osr.SpatialReference()
if np.average(lats) >= 0:
    hem_srs.ImportFromEPSG(3996)
else:
    hem_srs.ImportFromEPSG(3031)
print (hem_srs.ExportToWkt())

ll2hem = osr.CoordinateTransformation( ll_srs, hem_srs )
hem2ll = osr.CoordinateTransformation( hem_srs, ll_srs )

PROJCS["WGS 84 / IBCAO Polar Stereographic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",75],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",SOUTH],AXIS["Northing",SOUTH],AUTHORITY["EPSG","3996"]]


In [34]:
idx = 0

for time,age in zip(times,ages):
    dtstr = f"{time.strftime('%Y-%m-%d %H:%M:%S')}"

    lons = nc['lon'][:,idx]
    lats = nc['lat'][:,idx]
    zs = nc['z'][:,idx]
    # print (lons.shape,lats.shape,zs.shape)

    # Create point geometry collection
    multipoint = ogr.Geometry(ogr.wkbMultiPoint)

    # Loop through points and add to collection
    for lon,lat,z in zip(lons,lats,zs):
        point =  ogr.Geometry(ogr.wkbPoint)
        point.AddPoint( float(lon), float(lat), float(z) )
        multipoint.AddGeometry(point)
    # print (multipoint.GetGeometryCount())

    # Generate convex hull polygon
    concave_hull = multipoint.ConcaveHull(0.5,False)
    concave_hull.Transform( ll2hem )
    if age == 0 and ogr.GeometryTypeToName(concave_hull.GetGeometryType()) == '3D Point':
        print ('Buffering point')
        concave_hull = concave_hull.Buffer( 0.001 )
        concave_hull.SetCoordinateDimension(3)

    area = concave_hull.Area() / (1000.0*1000.0)
    perimeter = concave_hull.Boundary().Length() / (1000.0*1000.0)
    print (f"{idx:5d} {dtstr:19s} {age:15.1f}   Area = {area}   Perimeter = {perimeter}")

    concave_hull.Transform( hem2ll )

    # print("Concave Hull Geometry Type:", ogr.GeometryTypeToName(concave_hull.GetGeometryType()))
    # print("Concave Hull WKT:", concave_hull.ExportToWkt())

    # Create feature
    poly_feat = ogr.Feature(poly_lay.GetLayerDefn())

    # Add fields and geometry to polygon feature
    poly_feat.SetField( 'fid', idx )
    poly_feat.SetField( 'datetime', dtstr )
    poly_feat.SetField( 'age', float(age) )
    poly_feat.SetField( 'area', float(area) )
    poly_feat.SetField( 'perimeter', float(perimeter) )
    poly_feat.SetGeometry(concave_hull)

    # Add feature to file
    poly_lay.CreateFeature(poly_feat)

    # Dereference the feature
    poly_feat = None

    idx = idx + 1
    # if idx > 3:
    #     break

    0 2026-09-11 08:21:00             0.0   Area = 677.1153888103047   Perimeter = 0.20603027105173305
    1 2026-09-11 09:21:00          3600.0   Area = 610.9978303829613   Perimeter = 0.2061296469473349
    2 2026-09-11 10:21:00             nan   Area = 608.5628445798126   Perimeter = 0.14801284682882243
    3 2026-09-11 11:21:00             nan   Area = 588.4445781815028   Perimeter = 0.14552751156888505
    4 2026-09-11 12:21:00             nan   Area = 628.7729568063669   Perimeter = 0.13577687993999904
    5 2026-09-11 13:21:00             nan   Area = 627.2972776689611   Perimeter = 0.13564931001451086
    6 2026-09-11 14:21:00             nan   Area = 506.0038153470607   Perimeter = 0.13126002682424
    7 2026-09-11 15:21:00             nan   Area = 515.7753013070645   Perimeter = 0.13208734218887327
    8 2026-09-11 16:21:00             nan   Area = 522.5143479021625   Perimeter = 0.13299023909932087
    9 2026-09-11 17:21:00             nan   Area = 576.2631910357452   Perime

## Close output Shapefiles

In [43]:
pts_ds = None
lns_ds = None
poly_def = None
poly_ds = None

## Save and Download in other formats

In [53]:
import zipfile
import os
fn='iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_polygons'
path = "/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/"  # replace with your file name
base=path+fn
with zipfile.ZipFile(f"{path}shapefile_debris.zip", "w") as z:
    for ext in ["shp", "shx", "dbf", "prj"]:
        fname = f"{base}.{ext}"
        if os.path.exists(fname):
            z.write(fname)
            print(fname)

print(f"{path}shapefile_debris.zip")

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_polygons.shp
/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_polygons.shx
/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_polygons.dbf
/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/iceberg2026e_gebco_topaz5_windglophynrt_lead20_debris_end_results_polygons.prj
/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026/results/shapefile_debris.zip


In [54]:
import json

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "datetime": "2026-09-11T08:21:12.862463Z"
            },
            "geometry": {
                "type": "Point",
                "coordinates": [-29.18724, 66.87652]
            }
        }
    ]
}

with open("point.geojson", "w") as f:
    json.dump(geojson, f, indent=2)